# LeetCode 404 — Sum of Left Leaves bằng Python

**Link đề:** https://leetcode.com/problems/sum-of-left-leaves/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search (DFS), Breadth-First Search (BFS), Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sumOfLeftLeaves(self, root: TreeNode | None) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân** (binary tree) được lưu trong Python bằng class `TreeNode` như thế nào, và cách LeetCode ghi cây thành list `[3,9,20,null,null,15,7]`.
2. Phân biệt rõ **lá** (leaf), **lá trái** (left leaf) và "node nằm bên trái" — chỗ người mới hay hiểu sai nhất.
3. Viết được 4 lời giải: thu thập lá (baseline) → DFS đệ quy có cờ `is_left` → DFS "nhìn xuống con trái" → duyệt lặp bằng stack / BFS bằng queue.
4. Trace từng bước để thấy lời gọi đệ quy và stack thay đổi ra sao.
5. Chứng minh tính đúng, phân tích Time/Space theo `n` (số node) và `h` (chiều cao cây), tự giải biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tổng các lá trái

Cho `root` là gốc của một cây nhị phân. Trả về **tổng giá trị của mọi lá trái**.

- **Lá (leaf)**: node **không có con nào** (cả `left` và `right` đều là `None`).
- **Lá trái (left leaf)**: một lá **và** là **con trái** của một node khác.

| Input `root` | Output | Giải thích |
|---|:---:|---|
| `[3,9,20,null,null,15,7]` | `24` | lá trái là `9` (con trái của 3) và `15` (con trái của 20) |
| `[1]` | `0` | chỉ có gốc; gốc không là con của ai nên **không** phải lá trái |

**Constraints:**
- Số node trong khoảng `[1, 1000]`.
- `-1000 <= Node.val <= 1000` (giá trị có thể **âm**!).

Câu hỏi đặt ra trước khi code:
- Làm sao "đứng ở một node" mà biết nó là con trái hay con phải?
- Gốc có bao giờ là lá trái không?
- Cần đi qua mọi node không, hay có thể bỏ qua nhánh nào?

### 1.1. Vẽ cây ví dụ 1

List `[3,9,20,null,null,15,7]` là cách LeetCode ghi cây **theo từng tầng** (level order), `null` nghĩa là "chỗ này không có node":

```text
        3
       / \
      9   20
         /  \
        15   7
```

- `9`: con trái của 3, không có con → **lá trái** ✅
- `15`: con trái của 20, không có con → **lá trái** ✅
- `7`: là lá nhưng là **con phải** → không tính ❌
- `20`: là con phải và có con → không tính ❌

Tổng = `9 + 15 = 24`.

## 2. Nền tảng Python: class `TreeNode` và `None`

Một **class** là "khuôn" để tạo đối tượng. Mỗi `TreeNode` có 3 thuộc tính (attribute):

- `val`: giá trị của node,
- `left`: tham chiếu tới node con trái (hoặc `None` nếu không có),
- `right`: tham chiếu tới node con phải (hoặc `None`).

`None` trong Python là giá trị đặc biệt nghĩa là "không có gì". Kiểm tra bằng `x is None` / `x is not None`.

Trên LeetCode class này đã có sẵn; trong notebook ta tự định nghĩa để chạy độc lập.

In [1]:
from typing import Optional, List
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


# Tạo tay cây nhỏ:   1
#                   / \
#                  2   3
leaf2 = TreeNode(2)
leaf3 = TreeNode(3)
root = TreeNode(1, leaf2, leaf3)
print(root, "| con trái:", root.left, "| con phải:", root.right)
print("leaf2.left is None ->", leaf2.left is None)

TreeNode(1) | con trái: TreeNode(2) | con phải: TreeNode(3)
leaf2.left is None -> True


### 2.1. Dựng cây từ list level-order (giống LeetCode)

Để test nhanh, ta viết hàm `build_tree(values)` đọc list theo tầng:

1. Phần tử đầu là gốc, cho vào **hàng đợi** (queue).
2. Lấy lần lượt từng node ra khỏi queue, gán 2 phần tử kế tiếp trong list làm con trái, con phải (bỏ qua nếu là `None`).
3. Con mới tạo được cho vào cuối queue để lát nữa gán con cho nó.

`deque` (double-ended queue) trong `collections` cho phép `append` vào cuối và `popleft` ở đầu đều O(1). Ví dụ nhỏ trước:

In [2]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("queue:", list(q))
print("popleft ->", q.popleft())   # lấy phần tử vào sớm nhất
print("queue còn:", list(q))

queue: ['A', 'B', 'C']
popleft -> A
queue còn: ['B', 'C']


In [3]:
def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    # Dựng cây từ list level-order kiểu LeetCode (None thay cho null).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def show_tree(node: Optional[TreeNode], prefix: str = "", label: str = "root") -> None:
    # In cây nằm ngang: mỗi dòng một node, thụt lề theo độ sâu.
    if node is None:
        return
    is_leaf = node.left is None and node.right is None
    print(f"{prefix}{label}: {node.val}{'  (lá)' if is_leaf else ''}")
    show_tree(node.left, prefix + "    ", "L")
    show_tree(node.right, prefix + "    ", "R")


t1 = build_tree([3, 9, 20, None, None, 15, 7])
show_tree(t1)

root: 3
    L: 9  (lá)
    R: 20
        L: 15  (lá)
        R: 7  (lá)


### 2.2. Kiểm tra "lá" bằng một biểu thức

```python
node.left is None and node.right is None
```

`and` chỉ trả `True` khi **cả hai** vế đều đúng. Ta gói vào hàm nhỏ `is_leaf` để code dễ đọc.

In [4]:
def is_leaf(node: Optional[TreeNode]) -> bool:
    return node is not None and node.left is None and node.right is None

print("3 là lá?", is_leaf(t1))
print("9 là lá?", is_leaf(t1.left))
print("20 là lá?", is_leaf(t1.right))
print("15 là lá?", is_leaf(t1.right.left))
print("None là lá?", is_leaf(None))   # None không phải node -> False

3 là lá? False
9 là lá? True
20 là lá? False
15 là lá? True
None là lá? False


### 2.3. Đệ quy (recursion) trên cây

**Đệ quy** là hàm tự gọi lại chính nó trên bài toán nhỏ hơn. Với cây, "bài toán nhỏ hơn" chính là **cây con trái** và **cây con phải**.

Mỗi hàm đệ quy cần:
1. **Điều kiện dừng (base case)**: thường là `if node is None: return ...`.
2. **Bước đệ quy**: gọi lại cho `node.left`, `node.right` rồi **kết hợp** kết quả.

Ví dụ khởi động: đếm số node của cây.

In [5]:
def count_nodes(node: Optional[TreeNode]) -> int:
    if node is None:            # base case: cây rỗng có 0 node
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def sum_all(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return node.val + sum_all(node.left) + sum_all(node.right)

print("Số node:", count_nodes(t1))   # 5
print("Tổng mọi node:", sum_all(t1))  # 3+9+20+15+7 = 54

Số node: 5
Tổng mọi node: 54


**Nhận xét:** `sum_all` cộng **mọi** node. Bài 404 chỉ khác ở chỗ: chỉ cộng node thỏa "là lá" **và** "là con trái". Vấn đề là một node **không tự biết** nó là con trái hay con phải — thông tin đó nằm ở **cha** của nó. Đây là nút thắt của bài.

## 3. Cách trực giác (baseline) — thu thập mọi lá kèm "phía"

Ý tưởng dễ nghĩ nhất: đi khắp cây, mỗi khi gặp lá thì ghi lại `(giá trị, phía)` với phía là `"L"`, `"R"` hoặc `"root"`. Cuối cùng cộng các lá có phía `"L"`.

Cách này chưa tối ưu bộ nhớ (lưu cả danh sách lá) nhưng giúp **nhìn thấy dữ liệu** trước khi tính.

In [6]:
def collect_leaves(node: Optional[TreeNode], side: str, out: list) -> None:
    if node is None:
        return
    if node.left is None and node.right is None:
        out.append((node.val, side))
        return
    collect_leaves(node.left, "L", out)
    collect_leaves(node.right, "R", out)


def sum_left_leaves_baseline(root: Optional[TreeNode]) -> int:
    leaves = []
    collect_leaves(root, "root", leaves)
    print("  các lá:", leaves)
    return sum(val for val, side in leaves if side == "L")


print(sum_left_leaves_baseline(build_tree([3, 9, 20, None, None, 15, 7])))  # 24
print(sum_left_leaves_baseline(build_tree([1])))                            # 0

  các lá: [(9, 'L'), (15, 'L'), (7, 'R')]
24
  các lá: [(1, 'root')]
0


### 3.1. Cú pháp mới: generator expression trong `sum(...)`

```python
sum(val for val, side in leaves if side == "L")
```

Đọc là: "với mỗi cặp `(val, side)` trong `leaves`, nếu `side == "L"` thì lấy `val`, rồi cộng tất cả". Kết quả giống vòng `for` thường:

In [7]:
leaves = [(9, "L"), (15, "L"), (7, "R")]
total = 0
for val, side in leaves:        # tách cặp (tuple unpacking)
    if side == "L":
        total += val
print("vòng for:", total)
print("generator:", sum(val for val, side in leaves if side == "L"))

vòng for: 24
generator: 24


## 4. Nút thắt và insight

**Nút thắt:** đứng tại một node, ta không biết mình là con trái hay phải. Baseline giải quyết bằng cách lưu thêm danh sách lá — tốn bộ nhớ O(số lá) không cần thiết.

**Insight (2 cách nhìn tương đương):**

1. **Truyền thông tin xuống:** khi cha gọi đệ quy cho con trái, truyền kèm cờ `is_left=True`; con phải nhận `is_left=False`. Tại lá, chỉ cộng nếu `is_left` đúng.
2. **Nhìn xuống từ cha:** đứng ở node `p`, nếu `p.left` tồn tại và là lá thì cộng `p.left.val` ngay; còn lại đệ quy tiếp vào `p.left` (nếu không phải lá) và `p.right`.

Cả hai đều chỉ duyệt mỗi node một lần, không cần danh sách phụ.

In [8]:
# Minh họa "nhìn xuống từ cha" trên cây ví dụ 1
p = t1                      # node 3
print("p =", p.val, "| p.left =", p.left.val, "-> lá?", is_leaf(p.left))
p = t1.right                # node 20
print("p =", p.val, "| p.left =", p.left.val, "-> lá?", is_leaf(p.left))

p = 3 | p.left = 9 -> lá? True
p = 20 | p.left = 15 -> lá? True


## 5. Lời giải chính — DFS đệ quy với cờ `is_left`

Ta xây hàm `dfs(node, is_left)` trả về **tổng lá trái trong cây con gốc `node`**, biết rằng `node` là con trái của cha nếu `is_left=True`.

**Đoạn 1 — base case:** cây rỗng không có lá nào.
```python
if node is None:
    return 0
```

**Đoạn 2 — node là lá:** chỉ đóng góp khi nó là con trái.
```python
if node.left is None and node.right is None:
    return node.val if is_left else 0
```

**Đoạn 3 — node trong:** con trái nhận `True`, con phải nhận `False`, cộng hai kết quả.
```python
return dfs(node.left, True) + dfs(node.right, False)
```

**Khởi động:** gốc **không** phải con của ai → `dfs(root, False)`.

In [9]:
def sum_left_leaves_flag(root: Optional[TreeNode]) -> int:
    def dfs(node: Optional[TreeNode], is_left: bool) -> int:
        if node is None:
            return 0
        if node.left is None and node.right is None:
            return node.val if is_left else 0
        return dfs(node.left, True) + dfs(node.right, False)

    return dfs(root, False)


print(sum_left_leaves_flag(build_tree([3, 9, 20, None, None, 15, 7])))  # 24
print(sum_left_leaves_flag(build_tree([1])))                            # 0
print(sum_left_leaves_flag(build_tree([1, 2, 3, 4, 5])))                # 4

24
0
4


### 5.1. Cú pháp mới: hàm lồng (nested function) và biểu thức điều kiện

- `def dfs(...)` viết **bên trong** `sum_left_leaves_flag` gọi là hàm lồng: nó chỉ dùng được bên trong hàm ngoài, giúp gọn tên.
- `A if cond else B` là **biểu thức điều kiện** (ternary): trả `A` nếu `cond` đúng, ngược lại `B`.

In [10]:
is_left = True
print(9 if is_left else 0)
is_left = False
print(7 if is_left else 0)

9
0


## 6. TRACE từng bước — nhìn thấy lời gọi đệ quy

Hàm `trace_dfs` in ra mỗi lời gọi với độ thụt lề = độ sâu, để thấy thứ tự DFS "đi sâu trái trước".

Bảng trace mong đợi cho `[3,9,20,null,null,15,7]`:

| Lời gọi | node | is_left | Loại | Trả về |
|---|:---:|:---:|---|:---:|
| dfs(3, F) | 3 | False | node trong | 9 + 15 = 24 |
| dfs(9, T) | 9 | True | lá trái | 9 |
| dfs(20, F) | 20 | False | node trong | 15 + 0 = 15 |
| dfs(15, T) | 15 | True | lá trái | 15 |
| dfs(7, F) | 7 | False | lá phải | 0 |

In [11]:
def trace_dfs(root: Optional[TreeNode]) -> int:
    def dfs(node, is_left, depth):
        pad = "    " * depth
        if node is None:
            print(f"{pad}dfs(None) -> 0")
            return 0
        if node.left is None and node.right is None:
            res = node.val if is_left else 0
            kind = "lá TRÁI" if is_left else ("lá phải" if depth > 0 else "gốc là lá")
            print(f"{pad}dfs({node.val}, is_left={is_left}) : {kind} -> {res}")
            return res
        print(f"{pad}dfs({node.val}, is_left={is_left}) : node trong, đi xuống")
        left_sum = dfs(node.left, True, depth + 1)
        right_sum = dfs(node.right, False, depth + 1)
        res = left_sum + right_sum
        print(f"{pad}<- tại {node.val}: {left_sum} + {right_sum} = {res}")
        return res

    return dfs(root, False, 0)


print("Kết quả:", trace_dfs(build_tree([3, 9, 20, None, None, 15, 7])))

dfs(3, is_left=False) : node trong, đi xuống
    dfs(9, is_left=True) : lá TRÁI -> 9
    dfs(20, is_left=False) : node trong, đi xuống
        dfs(15, is_left=True) : lá TRÁI -> 15
        dfs(7, is_left=False) : lá phải -> 0
    <- tại 20: 15 + 0 = 15
<- tại 3: 9 + 15 = 24
Kết quả: 24


### 6.1. Trace nhánh đặc biệt: lá trái nằm dưới một **con phải**, và gốc là lá

Cây `[1,2,3,None,None,4]`:

```text
    1
   / \
  2   3
     /
    4
```

`4` là con trái của `3` (dù `3` là con phải của gốc) → vẫn là lá trái. Kết quả `2 + 4 = 6`. Còn cây `[1]` gốc là lá nhưng **không** được cộng.

In [12]:
print("Kết quả:", trace_dfs(build_tree([1, 2, 3, None, None, 4])))
print()
print("Kết quả:", trace_dfs(build_tree([1])))

dfs(1, is_left=False) : node trong, đi xuống
    dfs(2, is_left=True) : lá TRÁI -> 2
    dfs(3, is_left=False) : node trong, đi xuống
        dfs(4, is_left=True) : lá TRÁI -> 4
        dfs(None) -> 0
    <- tại 3: 4 + 0 = 4
<- tại 1: 2 + 4 = 6
Kết quả: 6

dfs(1, is_left=False) : gốc là lá -> 0
Kết quả: 0


## 7. Các lời giải khác (đi từ cơ bản lên)

### 7.1. Cách 3 — DFS "nhìn xuống con trái" (không cần cờ)

Tại node `p`:
- Nếu `p.left` là lá → cộng `p.left.val`, **không** cần đệ quy vào `p.left` nữa.
- Nếu `p.left` không phải lá → đệ quy vào `p.left`.
- Luôn đệ quy vào `p.right` (lá trái có thể nằm sâu trong nhánh phải).

In [13]:
def sum_left_leaves_lookdown(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    total = 0
    if root.left is not None:
        if root.left.left is None and root.left.right is None:
            total += root.left.val
        else:
            total += sum_left_leaves_lookdown(root.left)
    total += sum_left_leaves_lookdown(root.right)
    return total


for vals in ([3, 9, 20, None, None, 15, 7], [1], [1, 2, 3, None, None, 4]):
    print(vals, "->", sum_left_leaves_lookdown(build_tree(vals)))

[3, 9, 20, None, None, 15, 7] -> 24
[1] -> 0
[1, 2, 3, None, None, 4] -> 6


### 7.2. Cách 4 — DFS lặp bằng stack (tránh giới hạn đệ quy)

Python giới hạn độ sâu đệ quy (mặc định ~1000). Với `n <= 1000` thì một cây "lệch" hoàn toàn có thể chạm sát giới hạn. Dùng **stack** (ngăn xếp: vào sau ra trước, `list.append` + `list.pop`) để tự quản lý.

Mỗi phần tử stack là cặp `(node, is_left)`.

In [14]:
stack = []
stack.append("a"); stack.append("b"); stack.append("c")
print("pop ->", stack.pop(), "| còn", stack)   # c ra trước: LIFO

pop -> c | còn ['a', 'b']


In [15]:
def sum_left_leaves_stack(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    total = 0
    stack = [(root, False)]
    while stack:
        node, is_left = stack.pop()
        if node.left is None and node.right is None:
            if is_left:
                total += node.val
            continue
        if node.right is not None:
            stack.append((node.right, False))
        if node.left is not None:
            stack.append((node.left, True))
    return total


for vals in ([3, 9, 20, None, None, 15, 7], [1], [1, 2, 3, None, None, 4]):
    print(vals, "->", sum_left_leaves_stack(build_tree(vals)))

[3, 9, 20, None, None, 15, 7] -> 24
[1] -> 0
[1, 2, 3, None, None, 4] -> 6


### 7.3. Cách 5 — BFS theo tầng bằng queue

Thay stack bằng `deque` và `popleft` → duyệt theo **tầng** (level order). Logic kiểm tra giống hệt; chỉ khác thứ tự thăm. Kết quả tổng không phụ thuộc thứ tự cộng.

In [16]:
def sum_left_leaves_bfs(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    total = 0
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node.left is not None:
            if node.left.left is None and node.left.right is None:
                total += node.left.val
            else:
                queue.append(node.left)
        if node.right is not None:
            queue.append(node.right)
    return total


for vals in ([3, 9, 20, None, None, 15, 7], [1], [1, 2, 3, None, None, 4]):
    print(vals, "->", sum_left_leaves_bfs(build_tree(vals)))

[3, 9, 20, None, None, 15, 7] -> 24
[1] -> 0
[1, 2, 3, None, None, 4] -> 6


## 8. Code nộp LeetCode

Trên LeetCode **không** cần định nghĩa `TreeNode` (đã có sẵn). Chữ ký giữ đúng `sumOfLeftLeaves(self, root)`. Bên dưới là bản chính (DFS có cờ) và bản lặp (stack).

In [17]:
class Solution:
    def sumOfLeftLeaves(self, root: Optional[TreeNode]) -> int:
        def dfs(node, is_left):
            if node is None:
                return 0
            if node.left is None and node.right is None:
                return node.val if is_left else 0
            return dfs(node.left, True) + dfs(node.right, False)

        return dfs(root, False)


sol = Solution()
print(sol.sumOfLeftLeaves(build_tree([3, 9, 20, None, None, 15, 7])))  # 24
print(sol.sumOfLeftLeaves(build_tree([1])))                            # 0

24
0


In [18]:
class SolutionIterative:
    def sumOfLeftLeaves(self, root: Optional[TreeNode]) -> int:
        total = 0
        stack = [(root, False)] if root else []
        while stack:
            node, is_left = stack.pop()
            if not node.left and not node.right:
                total += node.val if is_left else 0
                continue
            if node.right:
                stack.append((node.right, False))
            if node.left:
                stack.append((node.left, True))
        return total


print(SolutionIterative().sumOfLeftLeaves(build_tree([3, 9, 20, None, None, 15, 7])))

24


## 9. Vì sao đúng? (invariant)

**Mệnh đề:** `dfs(node, is_left)` trả về tổng giá trị các lá trái **của cây gốc**, nằm trong cây con gốc `node`, với điều kiện `is_left` cho biết đúng `node` có phải con trái của cha nó hay không.

Chứng minh theo quy nạp trên kích thước cây con:
- `node is None`: không có lá → 0. Đúng.
- `node` là lá: lá này là lá trái **khi và chỉ khi** nó là con trái của cha → trả `node.val` nếu `is_left`, ngược lại 0. Đúng.
- `node` là node trong: mọi lá trong cây con nằm hoặc trong cây con trái, hoặc trong cây con phải (không chồng lấn). Con trái thật sự là con trái → truyền `True`; con phải → `False`. Theo giả thiết quy nạp hai lời gọi đúng → tổng đúng.

Gọi `dfs(root, False)` đúng vì gốc không có cha nên không thể là lá trái. **Không bỏ sót** (mọi node được thăm đúng một lần) và **không cộng nhầm** (chỉ cộng tại lá có cờ `True`).

## 10. Độ phức tạp

Gọi `n` là số node, `h` là chiều cao cây (`log n <= h <= n`).

| Cách | Time | Space phụ |
|---|:---:|:---:|
| Baseline thu thập lá | `O(n)` | `O(h)` đệ quy + `O(số lá)` danh sách |
| DFS đệ quy có cờ | `O(n)` | `O(h)` call stack |
| DFS nhìn xuống | `O(n)` | `O(h)` call stack |
| DFS lặp bằng stack | `O(n)` | `O(h)` stack (trong thực tế tối đa `O(n)` với cây xấu) |
| BFS bằng queue | `O(n)` | `O(w)` với `w` là độ rộng tầng lớn nhất (≤ `n`) |

- Cây cân bằng: `h ≈ log₂ n` → bộ nhớ nhỏ.
- Cây lệch thành "dây xích": `h = n` → đệ quy sâu `n` tầng.

Ai cũng phải thăm mọi node ít nhất một lần (lá trái có thể ở bất kỳ đâu), nên `O(n)` thời gian là tối ưu.

In [19]:
import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

# Cây lệch trái dài 1000 node: mỗi node chỉ có con trái
def build_left_chain(n):
    root = TreeNode(1)
    cur = root
    for v in range(2, n + 1):
        cur.left = TreeNode(v)
        cur = cur.left
    return root

chain = build_left_chain(1000)
print("Stack (lặp) trên dây 1000 node:", sum_left_leaves_stack(chain))   # chỉ lá cuối 1000

Giới hạn đệ quy mặc định: 1000
Stack (lặp) trên dây 1000 node: 1000


## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ đề: `[3,9,20,null,null,15,7] -> 24`, `[1] -> 0`.
2. Biên: chỉ có con phải là lá (`[1,null,2] -> 0`), chỉ có con trái là lá (`[1,2] -> 2`).
3. Phản ví dụ cho cách hiểu sai "cộng mọi node bên trái": `[1,2,3,4,5]` → chỉ `4` là lá trái (2 có con nên không tính) → `4`.
4. Giá trị âm: `[0,-5,3] -> -5`.
5. Lá trái nằm trong nhánh phải: `[1,2,3,None,None,4] -> 6`.
6. Đối chiếu ngẫu nhiên 5 cách trên nhiều cây.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<35} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", [3, 9, 20, None, None, 15, 7], 24),
    ("đề 2: chỉ gốc", [1], 0),
    ("chỉ có con phải là lá", [1, None, 2], 0),
    ("chỉ có con trái là lá", [1, 2], 2),
    ("node trái có con không tính", [1, 2, 3, 4, 5], 4),
    ("giá trị âm", [0, -5, 3], -5),
    ("lá trái trong nhánh phải", [1, 2, 3, None, None, 4], 6),
    ("cây đầy đủ 3 tầng", [1, 2, 3, 4, 5, 6, 7], 10),
]
for name, vals, exp in cases:
    check(name, sol.sumOfLeftLeaves(build_tree(vals)), exp)

PASS  đề 1                                -> 24
PASS  đề 2: chỉ gốc                       -> 0
PASS  chỉ có con phải là lá               -> 0
PASS  chỉ có con trái là lá               -> 2
PASS  node trái có con không tính         -> 4
PASS  giá trị âm                          -> -5
PASS  lá trái trong nhánh phải            -> 6
PASS  cây đầy đủ 3 tầng                   -> 10


In [21]:
import random

def random_tree(rng, n):
    # Chèn ngẫu nhiên n node: mỗi node mới gắn vào chỗ trống ngẫu nhiên.
    if n == 0:
        return None
    root = TreeNode(rng.randint(-1000, 1000))
    nodes = [root]
    for _ in range(n - 1):
        while True:
            parent = rng.choice(nodes)
            side = rng.choice(["left", "right"])
            if getattr(parent, side) is None:
                child = TreeNode(rng.randint(-1000, 1000))
                setattr(parent, side, child)
                nodes.append(child)
                break
    return root

def silent_baseline(root):
    leaves = []
    collect_leaves(root, "root", leaves)
    return sum(v for v, s in leaves if s == "L")

rng = random.Random(404)
funcs = [sum_left_leaves_flag, sum_left_leaves_lookdown, sum_left_leaves_stack,
         sum_left_leaves_bfs, Solution().sumOfLeftLeaves, SolutionIterative().sumOfLeftLeaves]
for _ in range(500):
    t = random_tree(rng, rng.randint(1, 40))
    expected = silent_baseline(t)
    for f in funcs:
        assert f(t) == expected
print("500 cây ngẫu nhiên: 6 cách đều khớp baseline -> PASS")

500 cây ngẫu nhiên: 6 cách đều khớp baseline -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng mọi node bên trái (không kiểm tra lá)
```python
def wrong1(node):
    if node is None: return 0
    s = node.left.val if node.left else 0     # cộng cả node trái có con!
    return s + wrong1(node.left) + wrong1(node.right)
```
**Vì sao sai:** đề chỉ tính **lá** trái. Với `[1,2,3,4,5]`, node `2` có con nhưng vẫn bị cộng.

**Sửa:** chỉ cộng khi `node.left.left is None and node.left.right is None`.

In [22]:
def wrong1(node):
    if node is None:
        return 0
    s = node.left.val if node.left else 0
    return s + wrong1(node.left) + wrong1(node.right)

t = build_tree([1, 2, 3, 4, 5])
print("sai:", wrong1(t), "| đúng:", Solution().sumOfLeftLeaves(t))

sai: 6 | đúng: 4


### Lỗi 2: Coi gốc là lá trái
```python
return dfs(root, True)    # sai: gốc không phải con trái của ai
```
Với `[1]` sẽ ra `1` thay vì `0`. **Sửa:** khởi động bằng `dfs(root, False)`.

### Lỗi 3: Chỉ đi vào nhánh trái, quên nhánh phải
Lá trái có thể nằm trong cây con **phải** (ví dụ `15` dưới `20`). Bỏ `dfs(node.right, False)` sẽ mất `15`.

In [23]:
def wrong2(root):
    def dfs(node, is_left):
        if node is None:
            return 0
        if node.left is None and node.right is None:
            return node.val if is_left else 0
        return dfs(node.left, True) + dfs(node.right, False)
    return dfs(root, True)          # lỗi 2

def wrong3(node):
    if node is None:
        return 0
    if node.left and not node.left.left and not node.left.right:
        return node.left.val
    return wrong3(node.left)        # lỗi 3: quên node.right

print("lỗi 2 với [1]:", wrong2(build_tree([1])), "(đúng là 0)")
print("lỗi 3 với ví dụ 1:", wrong3(build_tree([3, 9, 20, None, None, 15, 7])), "(đúng là 24)")

lỗi 2 với [1]: 1 (đúng là 0)
lỗi 3 với ví dụ 1: 9 (đúng là 24)


### Lỗi 4: Trả về sớm ở nhánh "nhìn xuống"
Trong `wrong3` ở trên còn một lỗi tinh vi: khi `node.left` là lá, hàm `return node.left.val` ngay và **bỏ qua cả nhánh phải**. Phải **cộng dồn** vào `total` rồi vẫn xét `node.right`.

### Lỗi 5: Truy cập `node.left.left` khi `node.left` là `None`
`None.left` gây `AttributeError`. Luôn kiểm tra `node.left is not None` trước.

In [24]:
try:
    n = TreeNode(1)
    print(n.left.left)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'left'


## 13. Mẹo nhận dạng pattern

- Đề hỏi về node theo **quan hệ với cha** (con trái, con phải, cháu, anh em…) → hoặc **truyền thông tin từ cha xuống** qua tham số (`is_left`, `parent`, `depth`), hoặc **đứng ở cha nhìn xuống**.
- Đề hỏi "tổng/đếm/tìm trên mọi node thỏa điều kiện" → DFS trả về giá trị, kết hợp `trái + phải`.
- Thấy cảnh báo cây có thể sâu → chuyển sang stack/queue.

**Template DFS truyền cờ:**
```python
def dfs(node, info_from_parent):
    if node is None: return 0
    if <là lá>: return <đóng góp dựa vào info>
    return dfs(node.left, <info cho con trái>) + dfs(node.right, <info cho con phải>)
```

## 14. Biến thể

1. **Tổng lá phải** — đổi vai trò `True/False`.
2. **Tổng mọi lá** — bỏ điều kiện `is_left`.
3. **Đếm lá trái** thay vì cộng giá trị.
4. **Tổng các node có ông (grandparent) chẵn** (LeetCode 1315) — truyền thêm `parent`, `grandparent`.
5. **Tổng lá ở tầng sâu nhất** (LeetCode 1302) — truyền `depth`.

In [25]:
def sum_right_leaves(root):
    def dfs(node, is_right):
        if node is None:
            return 0
        if node.left is None and node.right is None:
            return node.val if is_right else 0
        return dfs(node.left, False) + dfs(node.right, True)
    return dfs(root, False)

def sum_all_leaves(root):
    if root is None:
        return 0
    if root.left is None and root.right is None:
        return root.val
    return sum_all_leaves(root.left) + sum_all_leaves(root.right)

t = build_tree([3, 9, 20, None, None, 15, 7])
print("lá phải:", sum_right_leaves(t))   # 7
print("mọi lá:", sum_all_leaves(t))      # 9+15+7 = 31

lá phải: 7
mọi lá: 31


## 15. Bài tập (TODO — tự làm, không chạy hàm rỗng)

**Bài 1 (tính tay):** Với cây `[5,3,8,1,4,7,9,None,2]`, vẽ cây và liệt kê các lá trái. Tổng là bao nhiêu?

**Bài 2 (đoán output):** `Solution().sumOfLeftLeaves(build_tree([1, 2, None, 3, None, 4]))` in ra gì?

**Bài 3 (điền chỗ trống):** Hoàn thiện `exercise_sum_left(root)` theo kiểu DFS có cờ.

In [26]:
def exercise_sum_left(root):
    def dfs(node, is_left):
        # TODO 1: base case node None
        # TODO 2: nếu là lá -> trả node.val nếu is_left, ngược lại 0
        # TODO 3: đệ quy con trái (True) + con phải (False)
        pass
    return dfs(root, False)

# Tự làm xong rồi mới mở test:
# assert exercise_sum_left(build_tree([3, 9, 20, None, None, 15, 7])) == 24
# assert exercise_sum_left(build_tree([1])) == 0

**Bài 4 (tự viết):** Viết `exercise_count_left_leaves(root)` **đếm số lá trái** bằng BFS (queue), không dùng đệ quy.

In [27]:
def exercise_count_left_leaves(root):
    # TODO: dùng deque, đứng ở cha nhìn xuống con trái
    pass

# assert exercise_count_left_leaves(build_tree([3, 9, 20, None, None, 15, 7])) == 2
# assert exercise_count_left_leaves(build_tree([1])) == 0

**Bài 5 (biến thể):** Viết `exercise_sum_deepest_left_leaves(root)` trả về tổng các lá trái **ở độ sâu lớn nhất trong số các lá trái** (gốc có độ sâu 0). Nếu không có lá trái trả 0.

**Bài 6 (mini challenge):** Viết `exercise_left_leaves_paths(root)` trả về list các đường đi (list giá trị từ gốc tới lá) của mọi lá trái, theo thứ tự DFS trái trước.

In [28]:
def exercise_sum_deepest_left_leaves(root):
    # TODO: truyền thêm depth; nhớ max_depth và tổng tương ứng
    pass

def exercise_left_leaves_paths(root):
    # TODO: DFS mang theo path hiện tại
    pass

# assert exercise_sum_deepest_left_leaves(build_tree([3, 9, 20, None, None, 15, 7])) == 15
# assert exercise_left_leaves_paths(build_tree([3, 9, 20, None, None, 15, 7])) == [[3, 9], [3, 20, 15]]

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Vẽ cây theo tầng, đánh dấu `None` | Lá = không con; trái = con trái của cha | Lá trái là `1`? `2`? `7`? kiểm tra từng cái |
| 2 | Cây lệch trái dài | Mỗi node chỉ có con trái | Chỉ có 1 lá ở đáy |
| 3 | Theo mục 5 | 3 đoạn: None / lá / node trong | `return dfs(node.left, True) + dfs(node.right, False)` |
| 4 | Đứng ở cha nhìn xuống | `deque`, `popleft`, đếm `+= 1` | `if node.left and is_leaf(node.left): count += 1` |
| 5 | Cần biết độ sâu | `dfs(node, is_left, depth)` + biến ngoài | Nếu `depth > best`: reset tổng; `==`: cộng thêm |
| 6 | Mang theo đường đi | `path + [node.val]` khi đi xuống | Tại lá trái: `out.append(path)` |

## 17. Đáp án tham khảo

- **Bài 1:** cây
```text
          5
        /   \
       3     8
      / \   / \
     1   4 7   9
      \
       2
```
Lá: `2` (con **phải** của 1), `4` (phải), `7` (**trái** của 8), `9` (phải). `1` không phải lá. → tổng lá trái = **7**.
- **Bài 2:** cây `1 → 2 (trái) → 3 (trái) → 4 (trái)`; chỉ `4` là lá, và là con trái → **4**.

In [29]:
print("Bài 1:", Solution().sumOfLeftLeaves(build_tree([5, 3, 8, 1, 4, 7, 9, None, 2])))
print("Bài 2:", Solution().sumOfLeftLeaves(build_tree([1, 2, None, 3, None, 4])))

Bài 1: 7
Bài 2: 4


In [30]:
def answer_sum_left(root):
    def dfs(node, is_left):
        if node is None:
            return 0
        if node.left is None and node.right is None:
            return node.val if is_left else 0
        return dfs(node.left, True) + dfs(node.right, False)
    return dfs(root, False)


def answer_count_left_leaves(root):
    if root is None:
        return 0
    count = 0
    q = deque([root])
    while q:
        node = q.popleft()
        if node.left:
            if not node.left.left and not node.left.right:
                count += 1
            else:
                q.append(node.left)
        if node.right:
            q.append(node.right)
    return count


def answer_sum_deepest_left_leaves(root):
    best = [-1, 0]                    # [độ sâu tốt nhất, tổng]
    def dfs(node, is_left, depth):
        if node is None:
            return
        if node.left is None and node.right is None:
            if is_left:
                if depth > best[0]:
                    best[0], best[1] = depth, node.val
                elif depth == best[0]:
                    best[1] += node.val
            return
        dfs(node.left, True, depth + 1)
        dfs(node.right, False, depth + 1)
    dfs(root, False, 0)
    return best[1]


def answer_left_leaves_paths(root):
    out = []
    def dfs(node, is_left, path):
        if node is None:
            return
        path = path + [node.val]
        if node.left is None and node.right is None:
            if is_left:
                out.append(path)
            return
        dfs(node.left, True, path)
        dfs(node.right, False, path)
    dfs(root, False, [])
    return out


t = build_tree([3, 9, 20, None, None, 15, 7])
assert answer_sum_left(t) == 24 and answer_sum_left(build_tree([1])) == 0
assert answer_count_left_leaves(t) == 2 and answer_count_left_leaves(build_tree([1])) == 0
assert answer_sum_deepest_left_leaves(t) == 15
assert answer_sum_deepest_left_leaves(build_tree([1, 2, 3, 4, None, 5])) == 9   # 4 và 5 cùng sâu 2
assert answer_left_leaves_paths(t) == [[3, 9], [3, 20, 15]]
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sumOfLeftLeaves(self, root):
        def dfs(node, is_left):
            if not node:
                return 0
            if not node.left and not node.right:      # lá
                return node.val if is_left else 0
            return dfs(node.left, True) + dfs(node.right, False)
        return dfs(root, False)                         # gốc KHÔNG là lá trái
```

**Checklist trước khi submit:**
- [ ] Chỉ cộng **lá** (không con), không cộng node trái có con.
- [ ] Gốc khởi động với `is_left=False` (cây 1 node → 0).
- [ ] Vẫn đi vào nhánh **phải** (lá trái có thể nằm ở đó).
- [ ] Không truy cập `.left/.right` của `None`.
- [ ] Giá trị âm vẫn cộng bình thường (không dùng `0` làm "không có").
- [ ] Time `O(n)`, Space `O(h)`.